# 🏛️ Rajat Alpha V67: Institutional Weekly Swing Trading Radar & Screener (V3.5.1)
### **Senior Quantitative Strategist Architecture (15+ Years Multi-Cycle Experience)**
### **Dual Universe: Inside Major Indices vs. Broader Market Growth Alpha**
### **Configurable Watchlist Size (Default: 50 Setups Rated from A+ to B-)**

---

### **Strategic Overview & System Capabilities**
This institutional radar scans the broader US stock market (S&P 500, Nasdaq 100, Dow 30, and high-velocity mid-cap growth leaders) to produce a battle-tested swing trading watchlist for the upcoming week.

It directly aligns with the **`Single_Buy2.0` / `rajat_alpha_v67_single.py`** live execution engine by solving the fundamental swing trading problem: **separating merely extended stocks from actionable buy setups**.

#### **Key Architecture Components:**
1. **The 13 Core TradingView Screener Rules**:
   - `Price >= $10.00`, `Market Cap >= $2.0B`, `YTD Perf > +8.0%`
   - `EMA 21 >= SMA 50`, `EMA 10 >= EMA 21`, `SMA 50 >= SMA 150 >= SMA 200`
   - `Price >= 52w Low * 1.15`, `0% <= (Price - SMA50)/SMA50 <= 10%`, `(Price - SMA200)/SMA200 >= 10%`
   - `Volume Change > +10%` vs prior day or 21d avg, `Avg Vol (100d) > 250,000 shares`
2. **Dual Market Universe Segmentation**:
   - **List A: Inside Major Indices** (S&P 500, Nasdaq 100, Dow 30)
   - **List B: Outside Major Indices** (High-Growth Mid-Cap Leaders, Russell 1000/2000, IPO Leaders)
3. **Setup Horizon & Trading Readiness (The User's Core Need)**:
   - **TIER 1: TRIGGER-READY (0–3 Days / This Week)**: Already pulled back to 21 EMA / 50 SMA ($\le 2.0\%$), in Daily/Weekly Demand Zone ($\le 3.5\%$ from 21d low), ready for bot execution.
   - **TIER 2: ON-DECK (1–2 Weeks)**: Pulling back towards 21 EMA ($2.0\% - 4.2\%$), higher swing low ($SL_2 \ge SL_1$) established, coiling in Volatility Contraction Pattern (VCP).
   - **TIER 3: BASE FORMATION (2–3 Weeks)**: High-conviction leader consolidating after impulse breakout.
4. **5-Tier Letter Grade Rating (A+ to B-)**:
   - **Grade A+ (Elite)**: IQS $\ge 90$
   - **Grade A  (High Conviction)**: IQS $80 - 89$
   - **Grade B+ (Setup Developing)**: IQS $70 - 79$
   - **Grade B  (Standard Swing)**: IQS $60 - 69$
   - **Grade B- (Base Monitor / Reserve)**: IQS $50 - 59$
5. **Multi-Timeframe Demand Zone & Sell Zone Matrix**:
   - Evaluates Daily Demand Zone (Gate 5 in bot), Weekly Demand Zone (Gate 4), Monthly Demand Zone, and Overextended / Sell Zone.
6. **30-Day Swing Pivot Identification**:
   - Identifies $SL_1, SL_2$ (Swing Lows) and $SH_1, SH_2$ (Swing Highs) with exact prices, calendar dates, and contraction metrics.
7. **Volume Pocket Pivot & Accumulation Signature**:
   - Detects institutional volume surges on green candle closes.
8. **Automated Trade Execution Plan & Inline Candlestick Charts**:
   - Computes Suggested Entry, Stop Loss (ATR bounded 5–10%), Profit Targets (+5%, +10%, +15%), and Risk/Reward.
   - Renders 35-day daily candlestick charts with moving average overlays, shaded demand zones, and execution lines directly in the notebook!
9. **1-Click Downloads (Default: 50 Setups)**:
   - Formatted 6-Sheet Excel Workbook (`weekly_swing_screener_results.xlsx`).
   - Plain-text `watchlist.txt` ready to drop into `C:\Alpaca_Algo\Alpaca_Algo2.0\Single_Buy2.0\config\watchlist.txt`.
   - Direct symbol feed for `Rajat_Alpha_Institutional_Prop_Protocol.ipynb` and `Rajat_Alpha_v67_Colab_Backtest.ipynb`!


In [ ]:
# @title 📦 Step 1: Install Dependencies & Import Modules
!pip install --quiet yfinance openpyxl matplotlib pandas numpy tqdm

import os
import sys
import re
import urllib.request
import ssl
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

print("✅ Institutional Environment Initialized Successfully!")


In [ ]:
# @title ⚙️ Step 2: Configure Screener Parameters & Watchlist Size (Default: 50)
# @markdown Choose universe coverage, setup filters, and display options:

universe_mode = "Combined (Indices + Broader Growth)" #@param ["Combined (Indices + Broader Growth)", "Inside Major Indices (S&P 500, Nasdaq 100, Dow 30)", "Outside Major Indices (Broader Growth & Mid-Caps)", "Dow 30 Blue Chips", "Nasdaq 100 Leaders", "S&P 500 Core"]
output_excel_name = "weekly_swing_screener_results.xlsx" #@param {type:"string"}
output_watchlist_txt = "watchlist.txt" #@param {type:"string"}
top_watchlist_count = 50 #@param {type:"integer"}
render_candlestick_charts = True #@param {type:"boolean"}
max_charts_to_display = 10 #@param {type:"integer"}

mode_map = {
    "Combined (Indices + Broader Growth)": "combined",
    "Inside Major Indices (S&P 500, Nasdaq 100, Dow 30)": "indices",
    "Outside Major Indices (Broader Growth & Mid-Caps)": "broader",
    "Dow 30 Blue Chips": "dow30",
    "Nasdaq 100 Leaders": "nasdaq100",
    "S&P 500 Core": "sp500"
}
selected_mode = mode_map[universe_mode]

print(f"🎯 Universe Mode: {universe_mode}")
print(f"📊 Target Excel Export: {output_excel_name}")
print(f"📝 Target Watchlist TXT: {output_watchlist_txt}")
print(f"🔢 Watchlist Target Size: {top_watchlist_count} setups (Rated A+ to B-)")
print(f"📈 Chart Rendering: {render_candlestick_charts} (Max {max_charts_to_display} charts)")


In [ ]:
# @title 🌐 Step 3: Define Segmented Market Universes (Indices vs Broader Growth)

DOW_30_SYMBOLS = [
    "AAPL", "AMGN", "AMZN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS",
    "DOW", "GS", "HD", "HON", "IBM", "INTC", "JNJ", "JPM", "KO", "MCD",
    "MMM", "MRK", "MSFT", "NKE", "NVDA", "PG", "SHW", "TRV", "UNH", "V", "WMT"
]

NASDAQ_100_SYMBOLS = [
    "AAPL", "ABNB", "ADBE", "ADI", "ADP", "ADSK", "AEP", "AMAT", "AMD", "AMGN",
    "AMZN", "APP", "ARM", "ASML", "AVGO", "AXON", "BIIB", "BKNG", "BKR",
    "CCEP", "CDNS", "CDW", "CEG", "CHTR", "CPRT", "CRWD", "CSCO", "CSX", "CTAS",
    "CTSH", "DASH", "DDOG", "DLTR", "DXCM", "EA", "EXC", "FANG", "FAST", "FTNT",
    "GEHC", "GFS", "GILD", "GOOG", "GOOGL", "HON", "IDXX", "ILMN", "INTC", "INTU",
    "ISRG", "KDP", "KHC", "KLAC", "LIN", "LRCX", "LULU", "MAR", "MCHP", "MDB",
    "MDLZ", "MELI", "META", "MNST", "MRNA", "MRVL", "MSFT", "MU", "NFLX", "NVDA",
    "NXPI", "ODFL", "ON", "ORLY", "PANW", "PAYX", "PCAR", "PDD", "PEP", "PLTR",
    "PYPL", "QCOM", "REGN", "ROP", "ROST", "SBUX", "SNPS", "TEAM", "TMUS", "TSLA",
    "TTD", "TXN", "VRSK", "VRTX", "WBD", "WDAY", "XEL", "ZS"
]

SP_500_CORE_SAMPLE = [
    "AAPL", "NVDA", "MSFT", "AMZN", "META", "GOOGL", "GOOG", "BRK-B", "LLY", "AVGO",
    "TSLA", "JPM", "WMT", "UNH", "XOM", "V", "PG", "MA", "COST", "JNJ",
    "ORCL", "HD", "BAC", "ABBV", "CVX", "NFLX", "KO", "MRK", "CRM", "AMD",
    "PEP", "LIN", "TMO", "ACN", "WFC", "CSCO", "MCD", "ABT", "GE", "IBM",
    "PM", "ISRG", "TXN", "CAT", "NOW", "QCOM", "VZ", "MS", "AMAT", "UBER",
    "GS", "SPGI", "AXP", "BKNG", "RTX", "HON", "LOW", "PFE", "SYK", "UNP",
    "BLK", "TJX", "VRTX", "REGN", "ETN", "PANW", "PGR", "LRCX", "CB", "BSX",
    "ADI", "CI", "C", "MDT", "PLTR", "ANET", "KLAC", "SCHW", "SNPS",
    "CDNS", "CRWD", "MAR", "APH", "EOG", "TDG", "WM", "ITW", "SHW", "CTAS",
    "FTNT", "ORLY", "COF", "MCK", "NXPI", "DE", "WELL", "PH", "EMR", "PCAR"
]

BROADER_MARKET_SYMBOLS = [
    "NET", "SNOW", "HUBS", "ESTC", "PATH", "DOCN", "MSTR", "TOST", "DUOL", "IOT",
    "SYM", "TEM", "ALAB", "VRT", "RBRK", "TWLO", "SHOP", "SE", "BILL", "AFRM",
    "UPST", "HOOD", "SOFI", "COIN", "DT", "GWRE", "MNDY", "PEGA", "PCTY", "PAYC",
    "MANH", "TYL", "PTC", "APPF", "BL", "SMAR", "TENB", "VRNS", "OKTA", "FROG",
    "U", "RELY", "GH", "TSM", "SMCI", "PSTG", "CAHR", "MPWR", "ALGM", "COHR",
    "AMBA", "POWI", "RMBS", "ACLS", "DIOD", "LSCC", "MTSI", "TER", "ENTG", "MKSI",
    "CCMP", "FORM", "AMKR", "CAMT", "ICHR", "UCTT", "VECO", "INSW", "TNK", "LPG",
    "FRO", "STNG", "GSL", "ZIM", "VIST", "EQT", "LNG", "FSLR", "NXT", "AR",
    "RRC", "CHRD", "SM", "PR", "MTDR", "MGY", "CRK", "CNX", "DVN", "APA",
    "CTRA", "OVV", "HES", "OXY", "PAGP", "OKE", "PWR", "EME", "FIX", "BLDR",
    "IBP", "ACM", "J", "HUBB", "NVT", "GWW", "FDX", "SAIA", "XPO", "WSO",
    "TTC", "AGCO", "SITE", "TREX", "AZEK", "OSK", "CR", "RBC", "AIT", "GGG",
    "LECO", "MIDD", "CW", "HEI", "HII", "HWM", "ATI", "NDSN", "NSP", "TILE",
    "CELH", "ONON", "BIRK", "DECK", "SHAK", "CAVA", "WING", "BROS", "SG", "BOOT",
    "CROX", "SKX", "TSCO", "FIVE", "OLLI", "CHWY", "CART", "CAKE", "EAT", "DRI",
    "TXRH", "YUM", "DPZ", "PZZA", "BJRI", "PLAY", "LOCO", "ARCO", "DIN", "CHEF",
    "CNK", "IMAX", "ARGX", "ALNY", "BMRN", "CRSP", "BEAM", "NTRA", "RVMD", "KNSA",
    "AXSM", "INSM", "CYTK", "ARDX", "VKTX", "MDGL", "APGE", "HALO", "INCY", "NBIX",
    "JAZZ", "IONS", "ICPT", "ALKS", "PODD", "PEN", "TFX", "EW", "HOLX", "WST",
    "CAH", "BDX", "BMY", "IMVT", "GKOS", "PHVS", "AXGN", "THG", "KKR", "APO",
    "ARES", "BX", "OWL", "HLNE", "STEP", "CG", "LPLA", "RJF", "IBKR", "TW",
    "VIRT", "PIPR", "MC", "EVR", "CNO", "BFH", "UMBF", "WSBC", "BUSE", "PK",
    "RHI", "FCF", "NBTB", "BPOP", "S", "CBSH", "HNGE", "HSIC", "ACT", "NSIT",
    "USB", "WT", "PFG", "LFST", "UBSI", "AIZ", "ALL", "VOYA", "ONB", "ASH",
    "CLBK", "CATY", "BEN", "IVZ", "NTAP", "GCMG", "MET", "EG", "NTRS", "VCTR",
    "NMIH", "TNET", "NOV", "TIGO", "MSGE", "ECPG", "ROIV", "CLMT", "CON", "AVT",
    "XYZ", "RGA", "FA", "AMP", "KFY", "CXW"
]

def get_segmented_universe(mode: str = "combined") -> Tuple[List[str], Dict[str, str]]:
    m = mode.lower().strip()
    category_map = {}
    index_set = set(DOW_30_SYMBOLS + NASDAQ_100_SYMBOLS + SP_500_CORE_SAMPLE)
    
    if m in ["indices", "combined", "sp500"]:
        try:
            ctx = ssl._create_unverified_context()
            req = urllib.request.Request("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=ctx, timeout=5) as resp:
                html = resp.read().decode("utf-8")
            raw_syms = re.findall(r'href="https://www\.nyse\.com/quote/XNYS:([A-Z\.]+)"', html)
            if not raw_syms:
                raw_syms = re.findall(r'<td><a rel="nofollow" class="external text" href="[^"]+">([A-Z\.\-]+)</a></td>', html)
            if raw_syms:
                index_set.update([s.replace(".", "-") for s in raw_syms if len(s) <= 5])
        except Exception:
            pass
            
    for s in index_set: category_map[s] = "INDEX"
    broader_set = set(BROADER_MARKET_SYMBOLS) - index_set
    for s in broader_set: category_map[s] = "BROADER_MARKET"
    
    if m == "indices": final_list = sorted(list(index_set))
    elif m == "broader": final_list = sorted(list(broader_set))
    elif m == "dow30": final_list = sorted(list(set(DOW_30_SYMBOLS)))
    elif m == "nasdaq100": final_list = sorted(list(set(NASDAQ_100_SYMBOLS)))
    elif m == "sp500": final_list = sorted(list(index_set))
    else: final_list = sorted(list(index_set | broader_set))
    return final_list, category_map

universe_symbols, cat_map = get_segmented_universe(selected_mode)
print(f"Loaded Universe: {len(universe_symbols)} total equities")
print(f"  • Inside Major Indices: {sum(1 for s in universe_symbols if cat_map.get(s) == 'INDEX')}")
print(f"  • Outside Major Indices (Broader Growth): {sum(1 for s in universe_symbols if cat_map.get(s) == 'BROADER_MARKET')}")


In [ ]:
# @title 🧠 Step 4: Technical Engine, Demand Zone Classifier & A+ to B- Letter Grades

def calculate_screener_indicators(df: pd.DataFrame) -> Optional[pd.DataFrame]:
    if df is None or len(df) < 200: return None
    df = df.copy()
    c, h, l, v = df['Close'], df['High'], df['Low'], df['Volume']
    
    df['EMA10'] = c.ewm(span=10, adjust=False).mean()
    df['EMA21'] = c.ewm(span=21, adjust=False).mean()
    df['SMA50'] = c.rolling(window=50).mean()
    df['SMA150'] = c.rolling(window=150).mean()
    df['SMA200'] = c.rolling(window=200).mean()
    
    df['VOL_SMA21'] = v.rolling(window=21).mean()
    df['VOL_SMA100'] = v.rolling(window=100).mean()
    
    df['HIGH_52W'] = h.rolling(window=min(252, len(df))).max()
    df['LOW_52W'] = l.rolling(window=min(252, len(df))).min()
    df['LOW_21'] = l.rolling(window=21).min()
    
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    df['ATR14'] = tr.rolling(window=14).mean()
    
    df['SLOPE_EMA21'] = ((df['EMA21'] - df['EMA21'].shift(5)) / df['EMA21'].shift(5)) * 100.0
    df['SLOPE_SMA50'] = ((df['SMA50'] - df['SMA50'].shift(10)) / df['SMA50'].shift(10)) * 100.0
    df['SLOPE_SMA200'] = ((df['SMA200'] - df['SMA200'].shift(20)) / df['SMA200'].shift(20)) * 100.0
    return df

def calculate_multitimeframe_demand_zones(df: pd.DataFrame) -> Dict[str, Any]:
    curr = df.iloc[-1]
    close, low, high = float(curr['Close']), float(curr['Low']), float(curr['High'])
    ema21 = float(curr['EMA21'])
    low21 = float(curr['LOW_21']) if 'LOW_21' in curr and not pd.isna(curr['LOW_21']) else low
    
    daily_demand = low <= (low21 * 1.035)
    
    weekly_demand, weekly_bullish = False, True
    try:
        df_w = df.resample('W-FRI').agg({'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'}).dropna()
        if len(df_w) >= 21:
            w_c = float(df_w['Close'].iloc[-1])
            w_ema21 = float(df_w['Close'].ewm(span=21, adjust=False).mean().iloc[-1])
            w_low8 = float(df_w['Low'].iloc[-8:].min())
            weekly_bullish = w_c > w_ema21
            weekly_demand = (abs(w_c - w_ema21) / w_ema21 <= 0.03) or (w_c <= w_low8 * 1.05)
    except Exception:
        weekly_demand = daily_demand
        
    monthly_demand = False
    try:
        df_m = df.resample('ME').agg({'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'}).dropna()
        if len(df_m) >= 10:
            m_c = float(df_m['Close'].iloc[-1])
            m_ema10 = float(df_m['Close'].ewm(span=10, adjust=False).mean().iloc[-1])
            monthly_demand = (m_c > m_ema10) and (abs(m_c - m_ema10) / m_ema10 <= 0.06)
    except Exception:
        monthly_demand = True
        
    dist_ema21_pct = (close - ema21) / ema21 * 100.0
    c_range = max(high - low, 0.001)
    upper_wick = high - max(close, float(curr['Open']))
    is_overextended = dist_ema21_pct > 5.0
    is_shooting_star = (upper_wick / c_range >= 0.50) and (upper_wick / max(abs(close - float(curr['Open'])), 0.001) >= 2.0)
    
    sell_zone = "YES (Extended/Distribution Risk)" if (is_overextended or is_shooting_star) else "NO (Low-Risk Accumulation Area)"
    return {
        'daily_demand': "YES" if daily_demand else "NO",
        'weekly_demand': "YES" if weekly_demand else "NO",
        'monthly_demand': "YES" if monthly_demand else "NO",
        'weekly_bullish': weekly_bullish,
        'sell_zone': sell_zone
    }

def analyze_swing_structure(df: pd.DataFrame, window_bars: int = 30) -> Dict[str, Any]:
    if len(df) < window_bars + 2:
        return {'valid_structure': False, 'higher_low': False, 'contraction': False, 'sl1': 0.0, 'sl2': 0.0, 'sh1': 0.0, 'sh2': 0.0, 'sl1_date': 'N/A', 'sl2_date': 'N/A', 'sh1_date': 'N/A', 'sh2_date': 'N/A'}
    sub = df.iloc[-(window_bars + 2):].copy()
    highs, lows = [], []
    for i in range(1, len(sub) - 1):
        if sub['High'].iloc[i] > sub['High'].iloc[i-1] and sub['High'].iloc[i] > sub['High'].iloc[i+1]:
            highs.append((sub.index[i].strftime('%Y-%m-%d'), float(sub['High'].iloc[i])))
        if sub['Low'].iloc[i] < sub['Low'].iloc[i-1] and sub['Low'].iloc[i] < sub['Low'].iloc[i+1]:
            lows.append((sub.index[i].strftime('%Y-%m-%d'), float(sub['Low'].iloc[i])))
            
    higher_low, contraction = False, False
    sl1_val, sl2_val, sh1_val, sh2_val = 0.0, 0.0, 0.0, 0.0
    sl1_dt, sl2_dt, sh1_dt, sh2_dt = "N/A", "N/A", "N/A", "N/A"
    
    if len(lows) >= 2:
        sl1_dt, sl1_val = lows[-2]
        sl2_dt, sl2_val = lows[-1]
        higher_low = sl2_val >= (sl1_val * 0.995)
    if len(highs) >= 2:
        sh1_dt, sh1_val = highs[-2]
        sh2_dt, sh2_val = highs[-1]
        if sl1_val > 0 and sl2_val > 0:
            contraction = max(0.01, sh2_val - sl2_val) <= (max(0.01, sh1_val - sl1_val) * 1.05)
            
    return {
        'valid_structure': len(highs) >= 2 and len(lows) >= 2,
        'sh1': sh1_val, 'sh2': sh2_val, 'sl1': sl1_val, 'sl2': sl2_val,
        'sh1_date': sh1_dt, 'sh2_date': sh2_dt, 'sl1_date': sl1_dt, 'sl2_date': sl2_dt,
        'higher_low': higher_low, 'contraction': contraction
    }

def evaluate_candidate(symbol: str, df: pd.DataFrame, category: str = "INDEX", spy_df: Optional[pd.DataFrame] = None) -> Optional[Dict[str, Any]]:
    if df is None or len(df) < 200: return None
    c_bar, p_bar = df.iloc[-1], df.iloc[-2]
    close, open_px, prev_close = float(c_bar['Close']), float(c_bar['Open']), float(p_bar['Close'])
    volume, prev_volume = float(c_bar['Volume']), float(p_bar['Volume'])
    
    ema10, ema21 = float(c_bar['EMA10']), float(c_bar['EMA21'])
    sma50, sma150, sma200 = float(c_bar['SMA50']), float(c_bar['SMA150']), float(c_bar['SMA200'])
    vol_sma21 = float(c_bar['VOL_SMA21']) if not pd.isna(c_bar['VOL_SMA21']) else volume
    vol_sma100 = float(c_bar['VOL_SMA100']) if not pd.isna(c_bar['VOL_SMA100']) else volume
    high52w, low52w = float(c_bar['HIGH_52W']), float(c_bar['LOW_52W'])
    atr14 = float(c_bar['ATR14']) if not pd.isna(c_bar['ATR14']) else close * 0.03
    slope_ema21 = float(c_bar['SLOPE_EMA21']) if not pd.isna(c_bar['SLOPE_EMA21']) else 0.0
    slope_sma50 = float(c_bar['SLOPE_SMA50']) if not pd.isna(c_bar['SLOPE_SMA50']) else 0.0
    
    # 13 TRADINGVIEW CRITERIA
    if close < 10.0: return None
    if vol_sma100 < 250000: return None
    if ema21 < sma50: return None
    if (close - low52w) / low52w < 0.15: return None
    dist_sma50 = (close - sma50) / sma50
    if dist_sma50 < 0.0 or dist_sma50 > 0.10: return None
    dist_sma200 = (close - sma200) / sma200
    if dist_sma200 < 0.10: return None
    if sma50 < sma200: return None
    vol_chg_p = (volume - prev_volume) / prev_volume if prev_volume > 0 else 0.0
    vol_chg_m = (volume - vol_sma21) / vol_sma21 if vol_sma21 > 0 else 0.0
    if vol_chg_p <= 0.10 and vol_chg_m <= 0.10: return None
    if ema10 < ema21: return None
    if sma150 < sma200: return None
    if sma50 < sma150: return None
    
    # YTD Perf
    c_yr = df.index[-1].year
    ytd_bars = df[df.index.year == c_yr]
    ytd_perf = (close - float(ytd_bars['Open'].iloc[0])) / float(ytd_bars['Open'].iloc[0]) if len(ytd_bars) >= 2 else (close - float(df['Close'].iloc[-min(126, len(df))])) / float(df['Close'].iloc[-min(126, len(df))])
    if ytd_perf <= 0.08: return None
    
    # Senior Strategist Checks
    off_52w_high = (high52w - close) / high52w
    dist_ema21_pct = (close - ema21) / ema21 * 100.0
    dist_sma50_pct = (close - sma50) / sma50 * 100.0
    
    if abs(dist_ema21_pct) <= 2.0 and dist_sma50_pct <= 3.5: buy_zone = "Dual Convergence Sweet Spot"
    elif abs(dist_ema21_pct) <= 2.0: buy_zone = "21 EMA Prime Zone"
    elif dist_sma50_pct <= 3.0: buy_zone = "50 SMA Demand Zone"
    elif dist_ema21_pct <= 4.2: buy_zone = "Upper Pullback Zone"
    else: buy_zone = "Extended (>4.2%)"
    
    mtf = calculate_multitimeframe_demand_zones(df)
    swings = analyze_swing_structure(df, window_bars=30)
    
    # Setup Horizon
    if (abs(dist_ema21_pct) <= 2.0 or dist_sma50_pct <= 2.5) and (mtf['daily_demand'] == "YES" or mtf['weekly_demand'] == "YES") and slope_ema21 >= -0.2:
        horizon = "TIER 1: TRIGGER-READY (THIS WEEK / 0-3d)"
    elif dist_ema21_pct <= 4.2 and swings.get('higher_low', False):
        horizon = "TIER 2: ON-DECK (1-2 WEEKS)"
    else:
        horizon = "TIER 3: BASE FORMATION (2-3 WEEKS)"
        
    vol_surge_green = (vol_chg_p > 0.10 or vol_chg_m > 0.10) and (close > prev_close and close >= open_px)
    vol_tag = "YES (Green Pocket Pivot)" if vol_surge_green else ("YES (Volume Surge)" if (vol_chg_p > 0.10 or vol_chg_m > 0.10) else "NO")
    
    # IBD RS Rating
    rs_score = 50.0
    if spy_df is not None and len(spy_df) >= 200:
        try:
            stk_w = 0.4*(close/df['Close'].iloc[-63]-1) + 0.2*(df['Close'].iloc[-63]/df['Close'].iloc[-126]-1) + 0.2*(df['Close'].iloc[-126]/df['Close'].iloc[-189]-1) + 0.2*(df['Close'].iloc[-189]/df['Close'].iloc[-252]-1)
            spy_w = (spy_df['Close'].iloc[-1]/spy_df['Close'].iloc[-252]) - 1.0
            rs_score = round(max(1.0, min(99.0, 50.0 + (stk_w - spy_w)*150.0)), 1)
        except Exception: rs_score = 50.0
        
    # IQS Scoring (0-100)
    pts = 0
    if ema10 > ema21 > sma50 > sma150 > sma200: pts += 15
    if slope_ema21 > 0: pts += 8
    if slope_sma50 >= 0: pts += 7
    if buy_zone == "Dual Convergence Sweet Spot": pts += 25
    elif "21 EMA" in buy_zone: pts += 20
    elif "50 SMA" in buy_zone: pts += 18
    elif "Upper Pullback" in buy_zone: pts += 10
    else: pts += 2
    if swings.get('higher_low'): pts += 12
    if swings.get('contraction'): pts += 8
    rvol = volume / vol_sma21 if vol_sma21 > 0 else 1.0
    if rvol >= 1.5: pts += 10
    elif rvol >= 1.1: pts += 7
    else: pts += 4
    if close > prev_close: pts += 5
    if rs_score >= 80: pts += 6
    elif rs_score >= 65: pts += 4
    if off_52w_high <= 0.25: pts += 4
    if "TIER 1" in horizon: pts += 10
    elif "TIER 2" in horizon: pts += 5
    iqs = min(100, pts)
    
    # 5-Tier Letter Grade (A+ to B-)
    if iqs >= 90: grade = "A+"; stars = "⭐⭐⭐⭐⭐ (Grade A+ Elite)"
    elif iqs >= 80: grade = "A"; stars = "⭐⭐⭐⭐ (Grade A High Conviction)"
    elif iqs >= 70: grade = "B+"; stars = "⭐⭐⭐+ (Grade B+ Setup Developing)"
    elif iqs >= 60: grade = "B"; stars = "⭐⭐⭐ (Grade B Standard Swing)"
    else: grade = "B-"; stars = "⭐⭐ (Grade B- Base Monitor)"
    
    # Trade Plan
    entry = round(close if "TIER 1" in horizon else ema21, 2)
    sl = round(min(entry * 0.95, max(entry * 0.90, entry - 2.0 * atr14)), 2)
    risk_dlr = entry - sl
    risk_pct = round((risk_dlr / entry) * 100.0, 1) if entry > 0 else 7.0
    t1, t2, t3 = round(entry * 1.05, 2), round(entry * 1.10, 2), round(entry * 1.15, 2)
    rr = round((t2 - entry) / risk_dlr, 2) if risk_dlr > 0 else 1.6
    
    return {
        'symbol': symbol, 'category': category, 'setup_horizon': horizon, 'close': close,
        'ytd_perf': round(ytd_perf * 100, 1), 'rvol': round(rvol, 2), 'dist_ema21': round(dist_ema21_pct, 1),
        'dist_sma50': round(dist_sma50_pct, 1), 'slope_ema21': round(slope_ema21, 2), 'slope_sma50': round(slope_sma50, 2),
        'buy_zone': buy_zone, 'daily_demand': mtf['daily_demand'], 'weekly_demand': mtf['weekly_demand'],
        'monthly_demand': mtf['monthly_demand'], 'sell_zone': mtf['sell_zone'], 'vol_tag': vol_tag,
        'higher_low': 'YES' if swings.get('higher_low') else 'NO', 'contraction': 'YES' if swings.get('contraction') else 'NO',
        'sl1': round(swings.get('sl1', 0), 2), 'sl1_date': swings.get('sl1_date', 'N/A'),
        'sl2': round(swings.get('sl2', 0), 2), 'sl2_date': swings.get('sl2_date', 'N/A'),
        'sh1': round(swings.get('sh1', 0), 2), 'sh1_date': swings.get('sh1_date', 'N/A'),
        'sh2': round(swings.get('sh2', 0), 2), 'sh2_date': swings.get('sh2_date', 'N/A'),
        'rs_score': rs_score, 'iqs_score': iqs, 'grade': grade, 'star_rating': stars, 'entry': entry,
        'stop_loss': sl, 'risk_pct': risk_pct, 't1': t1, 't2': t2, 't3': t3, 'rr': rr
    }


In [ ]:
# @title 🚀 Step 5: Execute Institutional Scanning & Segmentation Pipeline

print(f"Scanning {len(universe_symbols)} equities ({selected_mode.upper()})...")

spy_raw = yf.download("SPY", period="2y", interval="1d", progress=False)
if isinstance(spy_raw.columns, pd.MultiIndex): spy_raw.columns = spy_raw.columns.get_level_values(0)
spy_df = calculate_screener_indicators(spy_raw.dropna())

results = []
stored_chart_data = {}
batch_size = 60

for b_start in tqdm(range(0, len(universe_symbols), batch_size), desc="Scanning Batches"):
    batch = universe_symbols[b_start:b_start + batch_size]
    try:
        data = yf.download(batch, period="2y", interval="1d", group_by="ticker", progress=False, threads=True)
        for sym in batch:
            try:
                df = data[sym].dropna(how="all") if len(batch) > 1 else data.dropna(how="all")
                if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
                if len(df) < 200: continue
                ind = calculate_screener_indicators(df)
                if ind is None: continue
                eval_res = evaluate_candidate(sym, ind, category=cat_map.get(sym, "INDEX"), spy_df=spy_df)
                if eval_res is not None:
                    results.append(eval_res)
                    if len(stored_chart_data) < max_charts_to_display:
                        stored_chart_data[sym] = (ind, eval_res)
            except Exception: pass
    except Exception: pass

results.sort(key=lambda x: (
    0 if "TIER 1" in x['setup_horizon'] else (1 if "TIER 2" in x['setup_horizon'] else 2),
    -x['iqs_score']
))

print("=" * 80)
print(f"🎯 Screening Complete! Total Qualifying Leaders: {len(results)}")
print(f"  • Tier 1 (Trigger-Ready This Week): {sum(1 for r in results if 'TIER 1' in r['setup_horizon'])}")
print(f"  • Tier 2 (On-Deck 1-2 Weeks): {sum(1 for r in results if 'TIER 2' in r['setup_horizon'])}")
print(f"  • Inside Major Indices: {sum(1 for r in results if r['category'] == 'INDEX')}")
print(f"  • Outside Major Indices (Broader Growth): {sum(1 for r in results if r['category'] == 'BROADER_MARKET')}")

grade_counts = {}
for c in results:
    g = c.get('grade', 'B')
    grade_counts[g] = grade_counts.get(g, 0) + 1
grade_summary = ", ".join([f"{g}: {cnt}" for g, cnt in sorted(grade_counts.items())])
print(f"  • Grade Breakdown (A+ to B-): {grade_summary}")
print("=" * 80)

# Display Top Results in Table (Configurable Default: 50 Setups)
df_top = pd.DataFrame(results[:top_watchlist_count])
if not df_top.empty:
    cols_show = ['symbol', 'category', 'setup_horizon', 'close', 'iqs_score', 'grade', 'star_rating', 'buy_zone', 'daily_demand', 'weekly_demand', 'vol_tag', 'entry', 'stop_loss', 'risk_pct', 't2', 'rr']
    display(df_top[cols_show])
else:
    print("No candidates passed all 13 rules in this batch.")


In [ ]:
# @title 📈 Step 6: Render Publication-Quality Candlestick Charts (One-Month Daily)

if render_candlestick_charts and stored_chart_data:
    for sym, (df_ind, cand) in list(stored_chart_data.items())[:max_charts_to_display]:
        sub = df_ind.iloc[-35:].copy()
        fig, ax = plt.subplots(figsize=(13, 6), facecolor='#0d1117')
        ax.set_facecolor('#161b22')
        
        width = 0.6
        up = sub[sub['Close'] >= sub['Open']]
        down = sub[sub['Close'] < sub['Open']]
        ax.bar(up.index, up['Close'] - up['Open'], width, bottom=up['Open'], color='#00e676', edgecolor='#00e676', zorder=3)
        ax.vlines(up.index, up['Low'], up['High'], color='#00e676', linewidth=1.2, zorder=2)
        ax.bar(down.index, down['Close'] - down['Open'], width, bottom=down['Open'], color='#ff5252', edgecolor='#ff5252', zorder=3)
        ax.vlines(down.index, down['Low'], down['High'], color='#ff5252', linewidth=1.2, zorder=2)
        
        ax.plot(sub.index, sub['EMA10'], color='#00e5ff', label='10 EMA', linewidth=1.4, zorder=4)
        ax.plot(sub.index, sub['EMA21'], color='#ffd700', label='21 EMA (Prime Support)', linewidth=2.0, zorder=4)
        ax.plot(sub.index, sub['SMA50'], color='#e040fb', label='50 SMA (Institutional Demand)', linewidth=1.8, zorder=4)
        if 'SMA200' in sub.columns and not pd.isna(sub['SMA200'].iloc[-1]):
            ax.plot(sub.index, sub['SMA200'], color='#ff1744', label='200 SMA (Macro Baseline)', linewidth=1.4, linestyle='--', zorder=4)
            
        low_21 = float(sub['Low'].min())
        ax.axhspan(low_21, low_21 * 1.035, color='#00e676', alpha=0.15, label='Daily Demand Zone (<=3.5% above 21d low)', zorder=1)
        
        if cand['sl1'] > 0:
            ax.scatter([sub.index[-12]], [cand['sl1']], color='#00e676', s=90, marker='^', zorder=5)
            ax.text(sub.index[-12], cand['sl1'] * 0.985, f"SL1: ${cand['sl1']:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
        if cand['sl2'] > 0:
            ax.scatter([sub.index[-4]], [cand['sl2']], color='#00e676', s=110, marker='^', zorder=5)
            ax.text(sub.index[-4], cand['sl2'] * 0.985, f"SL2: ${cand['sl2']:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
        if cand['sh1'] > 0:
            ax.scatter([sub.index[-18]], [cand['sh1']], color='#ff5252', s=90, marker='v', zorder=5)
            ax.text(sub.index[-18], cand['sh1'] * 1.01, f"SH1: ${cand['sh1']:.2f}", color='#ff5252', fontsize=8, ha='center', weight='bold')
        if cand['sh2'] > 0:
            ax.scatter([sub.index[-8]], [cand['sh2']], color='#ff5252', s=110, marker='v', zorder=5)
            ax.text(sub.index[-8], cand['sh2'] * 1.01, f"SH2: ${cand['sh2']:.2f}", color='#ff5252', fontsize=8, ha='center', weight='bold')
            
        ax.axhline(cand['entry'], color='#00e5ff', linestyle='--', linewidth=1.5, label=f"Projected Entry: ${cand['entry']:.2f}")
        ax.axhline(cand['stop_loss'], color='#ff1744', linestyle='--', linewidth=1.5, label=f"Stop Loss: ${cand['stop_loss']:.2f} (-{cand['risk_pct']}%)")
        ax.axhline(cand['t1'], color='#69f0ae', linestyle=':', linewidth=1.2, label=f"Target 1 (+5%): ${cand['t1']:.2f}")
        ax.axhline(cand['t2'], color='#ffd700', linestyle=':', linewidth=1.2, label=f"Target 2 (+10%): ${cand['t2']:.2f} (R:R {cand['rr']}:1)")
        
        ax.grid(True, color='#21262d', linestyle='--', alpha=0.6)
        ax.tick_params(colors='#8b949e', labelsize=9)
        for spine in ax.spines.values(): spine.set_color('#30363d')
        
        plt.suptitle(f"{sym} [{cand['category']}] — {cand['setup_horizon']} | Grade: {cand['grade']} (IQS: {cand['iqs_score']}/100)", color='#f0f6fc', fontsize=12, fontweight='bold', y=0.98)
        plt.title(f"Buy Zone: {cand['buy_zone']} | Daily Demand: {cand['daily_demand']} | Weekly Demand: {cand['weekly_demand']} | Vol Pocket Pivot: {cand['vol_tag']}", color='#8b949e', fontsize=9, pad=8)
        ax.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', loc='upper left', fontsize=8, ncol=2)
        plt.show()
        plt.close()
else:
    print("Chart rendering skipped or no candidates available.")


In [ ]:
# @title 📊 Step 7: Export 6-Sheet Institutional Excel Workbook & watchlist.txt (Configurable Top N, Default: 50)

wb = openpyxl.Workbook()
wb.remove(wb.active)

navy_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
gold_fill = PatternFill(start_color="B28900", end_color="B28900", fill_type="solid")
green_fill = PatternFill(start_color="1E4620", end_color="1E4620", fill_type="solid")
purple_fill = PatternFill(start_color="4A154B", end_color="4A154B", fill_type="solid")
slate_fill = PatternFill(start_color="333F48", end_color="333F48", fill_type="solid")

h_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
reg_font = Font(name="Calibri", size=10)
thin_b = Border(left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'), top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9'))

headers = [
    "Rank", "Symbol", "Category", "Setup Horizon", "Price ($)", "IQS Score", "Grade", "Star Rating",
    "Buy Zone", "Dist EMA21 (%)", "Dist SMA50 (%)", "EMA21 Slope", "SMA50 Slope",
    "Daily Demand", "Weekly Demand", "Volume Pocket Pivot", "SL2 >= SL1", "VCP Contraction",
    "SL1 ($)", "SL2 ($)", "SH1 ($)", "SH2 ($)", "IBD RS", "Projected Entry ($)", "Stop Loss ($)",
    "Risk (%)", "PT1 (+5%)", "PT2 (+10%)", "PT3 (+15%)", "Risk/Reward"
]

def write_sheet(ws, cands, fill):
    ws.append(headers)
    for i in range(1, len(headers)+1):
        c = ws.cell(1, i); c.fill = fill; c.font = h_font; c.alignment = Alignment(horizontal="center")
    for idx, r in enumerate(cands, 1):
        ws.append([
            idx, r['symbol'], r['category'], r['setup_horizon'], r['close'], r['iqs_score'],
            r.get('grade', 'B'), r['star_rating'], r['buy_zone'], f"{r['dist_ema21']:+.1f}%", f"{r['dist_sma50']:+.1f}%",
            r['slope_ema21'], r['slope_sma50'], r['daily_demand'], r['weekly_demand'],
            r['vol_tag'], r['higher_low'], r['contraction'], r['sl1'], r['sl2'], r['sh1'], r['sh2'],
            r['rs_score'], r['entry'], r['stop_loss'], f"{r['risk_pct']:.1f}%", r['t1'], r['t2'], r['t3'], r['rr']
        ])
        for col_idx in range(1, len(headers)+1):
            cell = ws.cell(ws.max_row, col_idx); cell.font = reg_font; cell.border = thin_b

idx_cands = [r for r in results if r['category'] == "INDEX"]
brd_cands = [r for r in results if r['category'] == "BROADER_MARKET"]
t1_cands = [r for r in results if "TIER 1" in r['setup_horizon']]

# 1. Index Top Setups (Top N)
ws1 = wb.create_sheet("Index_Watchlist_Top_Setups")
write_sheet(ws1, idx_cands[:top_watchlist_count], navy_fill)

# 2. Broader Market Alpha Setups (Top N)
ws2 = wb.create_sheet("Broader_Market_Alpha_Setups")
write_sheet(ws2, brd_cands[:top_watchlist_count], gold_fill)

# 3. Trigger Ready This Week (Top N)
ws3 = wb.create_sheet("Trigger_Ready_This_Week")
write_sheet(ws3, t1_cands[:top_watchlist_count] if t1_cands else results[:top_watchlist_count], green_fill)

# 4. Multi-Timeframe Demand Matrix
ws4 = wb.create_sheet("Multi_Timeframe_Demand_Matrix")
h_d = ["Rank", "Symbol", "Category", "Setup Horizon", "Price ($)", "Grade", "Daily Demand", "Weekly Demand", "Monthly Demand", "Sell Zone", "Dist EMA21 (%)", "Dist SMA50 (%)", "Volume Surge Tag", "SL2 >= SL1", "VCP Contraction", "IQS Score"]
ws4.append(h_d)
for i in range(1, len(h_d)+1):
    c = ws4.cell(1, i); c.fill = purple_fill; c.font = h_font; c.alignment = Alignment(horizontal="center")
for idx, r in enumerate(results, 1):
    ws4.append([idx, r['symbol'], r['category'], r['setup_horizon'], r['close'], r.get('grade', 'B'), r['daily_demand'], r['weekly_demand'], r['monthly_demand'], r['sell_zone'], r['dist_ema21'], r['dist_sma50'], r['vol_tag'], r['higher_low'], r['contraction'], r['iqs_score']])

# 5. All Screened Candidates
ws5 = wb.create_sheet("All_Screened_Candidates")
write_sheet(ws5, results, slate_fill)

# 6. Watchlist TXT Export (Top N)
ws6 = wb.create_sheet("Watchlist_TXT_Export")
ws6.append(["Symbol (Ready to Copy to config/watchlist.txt)", "Category", "Setup Horizon", "IQS Score", "Grade"])
for i in range(1, 6): ws6.cell(1, i).fill = navy_fill; ws6.cell(1, i).font = h_font
for r in results[:top_watchlist_count]:
    ws6.append([r['symbol'], r['category'], r['setup_horizon'], r['iqs_score'], r.get('grade', 'B')])

for ws in [ws1, ws2, ws3, ws4, ws5, ws6]:
    for col in ws.columns:
        max_l = max(len(str(c.value or '')) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = max(max_l + 3, 12)

wb.save(output_excel_name)
print(f"✅ Excel workbook saved ({top_watchlist_count} setups per sheet): {output_excel_name}")

# Plain-text watchlist.txt (Top N)
with open(output_watchlist_txt, "w") as f:
    for r in results[:top_watchlist_count]:
        f.write(f"{r['symbol']}\n")
print(f"✅ Plain-text watchlist saved ({top_watchlist_count} setups): {output_watchlist_txt}")

# 1-Click Downloads
try:
    from google.colab import files
    files.download(output_excel_name)
    files.download(output_watchlist_txt)
    print("📥 Download dialogs triggered in your browser for Excel and watchlist.txt!")
except Exception:
    print(f"Files saved locally as: {output_excel_name} and {output_watchlist_txt}")


In [ ]:
# @title 🔗 Step 8: Direct Backtest Colab Integration (1-Click Paste)
# @markdown Copy this symbol list and instructions to test portfolio compounding in `Rajat_Alpha_Institutional_Prop_Protocol.ipynb` or `Rajat_Alpha_v67_Colab_Backtest.ipynb`:

top_syms = [r['symbol'] for r in results[:top_watchlist_count]]
formatted_py_list = str(top_syms)

print("=" * 80)
print(f"🎯 HOW TO RUN IMMEDIATE BACKTEST ON THIS SCREENED WATCHLIST ({len(top_syms)} SETUPS RATED A+ TO B-):")
print("=" * 80)
print("1. Open 'Rajat_Alpha_v67_Colab_Backtest.ipynb' or 'Rajat_Alpha_Institutional_Prop_Protocol.ipynb'")
print("2. In the 'Symbols Configuration' cell, paste the following list:")
print()
print(f"SCANNED_WATCHLIST = {formatted_py_list}")
print()
print("3. Run the backtest cell to see historical Sharpe, Win Rate, Profit Factor, and Equity Curve!")
print("4. Or drop the downloaded 'watchlist.txt' directly into:")
print("   C:\\Alpaca_Algo\\Alpaca_Algo2.0\\Single_Buy2.0\\config\\watchlist.txt")
print("=" * 80)
